# 02 · Current LSI Reproduction

**Purpose.** Воспроизвести текущую Global/Local модель через production-код и сверить с сохранёнными scores; посмотреть стресс-эпизоды на production-шкале.

**What to look for.**
- совпадает ли переобученный Global с production lsi_scores
- значения Global в Dec2014 / Feb2022 / Aug2023 (production-шкала)
- окно Local и его покрытие
- текущие пороги и статусы

> Это lab-ноутбук: выводы здесь предварительные, не финальный отчёт. Меняй параметры в ячейке *Parameters* и перезапускай.

In [ ]:
# --- bootstrap: запуск из корня проекта (рядом с data/ и backend/) ---
import sys, os
from pathlib import Path
# найти корень проекта и встать в него
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'data' / 'processed').is_dir()), _here)
os.chdir(_root)
sys.path.insert(0, str(_root))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 60)
from lab import utils as u
print('project root:', _root.name)

### Обучение через production-код backend

In [ ]:
from backend.src.services.lsi_training_service import fit_lsi_artifact, build_lsi_models
d = u.load_final_dataset()
g_art, g_scores = fit_lsi_artifact(d, kind='global')
print('global rows', len(g_scores), 'lsi range', g_scores['lsi_global'].min(), g_scores['lsi_global'].max())

### Сверка с сохранёнными production scores

In [ ]:
saved = u.load_lsi_scores()
merged = g_scores[['date','lsi_global']].merge(saved[['date','lsi_global']], on='date', suffixes=('_refit','_saved'))
cmp = u.compare_scores(merged['lsi_global_refit'], merged['lsi_global_saved'])
print(cmp)

### График Global (вся история) с порогами и эпизодами

In [ ]:
fig, ax = u.plot_lsi_series(g_scores['date'], {'Global (refit)': g_scores['lsi_global']},
    episodes=u.STRESS_EPISODES, thresholds=u.DEFAULT_THRESHOLDS, title='Global LSI — production scale')
plt.show()

### Стресс-эпизоды на production-шкале
Важно: это full-history production-шкала, НЕ backtest-шкала (см. 04).

In [ ]:
u.compute_episode_summary(g_scores['date'], g_scores['lsi_global'])

### Local: окно и покрытие

In [ ]:
loc = saved[saved['lsi_local'].notna()]
print('local non-null rows:', len(loc))
if len(loc):
    print('window:', loc['date'].min().date(), '->', loc['date'].max().date())
fig, ax = u.plot_lsi_series(saved['date'], {'Local': saved['lsi_local'], 'Global': saved['lsi_global']},
    thresholds=u.DEFAULT_THRESHOLDS, title='Local vs Global (saved scores)'); plt.show()

### Текущие пороги и распределение статусов

In [ ]:
from backend.src.services.lsi_thresholds import get_lsi_status, LSI_THRESHOLD_PROFILES
print('profiles:', list(LSI_THRESHOLD_PROFILES))
g_scores['status'] = g_scores['lsi_global'].map(get_lsi_status)
g_scores['status'].value_counts()

## Notes / Open questions

- Refit совпадает с saved? Если max_abs_diff велик — sklearn version mismatch в pickled артефакте.
- Dec2014 на production-шкале остаётся YELLOW — это ключевой факт для 04.
- Local покрывает только ~последние 365 дней — мало для валидации (см. 06).